# LeetCode 1441 — Build an Array With Stack Operations bằng Python

Link đề: https://leetcode.com/problems/build-an-array-with-stack-operations/ · Độ khó: **Medium** · Chủ đề: Array, Stack, Simulation

Skeleton LeetCode:

```python
class Solution:
    def buildArray(self, target: list[int], n: int) -> list[str]:
        ...
```

## 1. Mục tiêu học

1. Hiểu **stack** (ngăn xếp) với hai thao tác `Push` / `Pop` và cách dùng `list` của Python làm stack (`append`, `pop`).
2. Biết **mô phỏng** (simulation) một quy trình theo đúng luật đề, trước hết bằng cách "làm thật" rồi rút gọn.
3. Nhận ra quy tắc then chốt: số **có** trong `target` ⇒ `"Push"`; số **không có** ⇒ `"Push","Pop"`; và **dừng ngay** khi đã đọc tới `target[-1]`.
4. Viết lời giải theo **khoảng trống (gap)** giữa các phần tử `target` và tính được độ dài kết quả `2·target[-1] − len(target)`.
5. Tránh lỗi: chạy tới `n` thay vì dừng sớm, `ops += "Push"` tách thành ký tự, quên `"Push"` trước `"Pop"`.

## 2. Đề bài & ví dụ

Cho mảng `target` (tăng **nghiêm ngặt**) và số nguyên `n`. Có một stack rỗng với 2 thao tác:
- `"Push"`: đẩy một số lên đỉnh stack.
- `"Pop"`: bỏ số ở đỉnh stack.

Có một **dòng số (stream)** gồm các số `1, 2, …, n` theo thứ tự. Luật chơi:
- Nếu stream còn số, có thể lấy **số kế tiếp** và `Push` nó.
- Nếu stack không rỗng, có thể `Pop` đỉnh.
- **Ngay khi** stack (từ đáy lên đỉnh) bằng `target` thì **dừng**: không đọc thêm số nào, không thao tác thêm.

Trả về danh sách thao tác để xây được `target`. Nếu có nhiều đáp án, trả về bất kỳ.

**Ví dụ 1:** `target = [1,3], n = 3` → `["Push","Push","Pop","Push"]` (đẩy 1, đẩy 2, bỏ 2, đẩy 3).

**Ví dụ 2:** `target = [1,2,3], n = 3` → `["Push","Push","Push"]`.

**Ví dụ 3:** `target = [1,2], n = 4` → `["Push","Push"]`. Đáp án đọc thêm số 3 **không được chấp nhận** vì phải dừng khi stack đã bằng `target`.

**Ràng buộc:** `1 <= target.length <= 100`, `1 <= n <= 100`, `1 <= target[i] <= n`, `target` tăng nghiêm ngặt.

In [1]:
import random
import time
from typing import List

examples = [
    (([1, 3], 3), ["Push", "Push", "Pop", "Push"]),
    (([1, 2, 3], 3), ["Push", "Push", "Push"]),
    (([1, 2], 4), ["Push", "Push"]),
]
for (target, n), want in examples:
    print(f"target={target}, n={n}  ->  {want}")

target=[1, 3], n=3  ->  ['Push', 'Push', 'Pop', 'Push']
target=[1, 2, 3], n=3  ->  ['Push', 'Push', 'Push']
target=[1, 2], n=4  ->  ['Push', 'Push']


## 3. Kiến thức nền

### 3.1 Dùng `list` làm stack

Đỉnh stack là **cuối list**. `append(x)` = Push, `pop()` = Pop, `stack[-1]` = xem đỉnh. Cả ba đều `O(1)`.

In [2]:
stack = []
stack.append(1)      # Push 1
stack.append(2)      # Push 2
print("sau 2 lần Push:", stack, "đỉnh =", stack[-1])
top = stack.pop()    # Pop
print("Pop ra", top, "->", stack)

sau 2 lần Push: [1, 2] đỉnh = 2
Pop ra 2 -> [1]


### 3.2 `range(1, n + 1)` — dòng số 1..n

`range(a, b)` không lấy `b`, nên dòng số `1..n` là `range(1, n + 1)`. Có thể dùng `iter(...)` và `next(...)` để "đọc số kế tiếp" giống stream thật.

In [3]:
stream = iter(range(1, 5))
print(next(stream), next(stream), next(stream))
print(list(stream), "<- phần còn lại")

1 2 3
[4] <- phần còn lại


### 3.3 `append`, `extend`, `+=` và nhân list

- `ops.append("Push")` thêm **một** phần tử.
- `ops.extend(["Push", "Pop"])` hoặc `ops += ["Push", "Pop"]` thêm **từng phần tử** của list.
- `["Push", "Pop"] * 3` lặp list 3 lần (phẳng).
- Cẩn thận: `ops += "Push"` coi chuỗi là dãy ký tự ⇒ thêm `'P','u','s','h'`!

In [4]:
ops = []
ops.append("Push")
ops.extend(["Push", "Pop"])
ops += ["Push"]
print(ops)
print(["Push", "Pop"] * 2)
bad = []
bad += "Push"
print("ops += 'Push' cho:", bad)

['Push', 'Push', 'Pop', 'Push']
['Push', 'Pop', 'Push', 'Pop']
ops += 'Push' cho: ['P', 'u', 's', 'h']


### 3.4 Kiểm tra thuộc tập: `list` vs `set`

`x in list` duyệt tuần tự `O(m)`; `x in set` trung bình `O(1)`. Với `m ≤ 100` không khác biệt nhiều, nhưng đây là thói quen tốt.

In [5]:
target = [1, 3, 7]
tset = set(target)
print([x for x in range(1, 8) if x in tset])
print([x for x in range(1, 8) if x not in tset])

[1, 3, 7]
[2, 4, 5, 6]


### 3.5 So sánh hai list bằng `==`

`[1, 3] == [1, 3]` so từng phần tử theo thứ tự — tiện để kiểm tra "stack có bằng target chưa".

In [6]:
print([1, 3] == [1, 3], [1, 3] == [3, 1], [1] == [1, 3])

True False False


## 4. Ý tưởng nền tảng / trực giác

Stream chỉ đi **tới**, không bao giờ quay lại. Vì `target` tăng dần, mỗi số `x` đọc được chỉ có 2 số phận:
- `x` **có trong** `target` ⇒ phải **giữ**: chỉ `"Push"`.
- `x` **không có** ⇒ không được để lại trong stack: `"Push"` rồi `"Pop"` ngay (vì số kế tiếp sẽ đè lên, sau này không bỏ được nữa mà không bỏ luôn phần trên).

Tính tay `target = [2,3,5]`, `n = 6`:

| số đọc | có trong target? | thao tác | stack |
|---|---|---|---|
| 1 | không | Push, Pop | [] |
| 2 | có | Push | [2] |
| 3 | có | Push | [2,3] |
| 4 | không | Push, Pop | [2,3] |
| 5 | có | Push | [2,3,5] = target → **dừng** |

Số 6 không bao giờ được đọc. Kết quả: `Push Pop Push Push Push Pop Push` (7 thao tác = `2·5 − 3`).

In [7]:
target, n = [2, 3, 5], 6
tset, stack, ops = set(target), [], []
for x in range(1, n + 1):
    if stack == target:
        break
    if x in tset:
        ops.append("Push"); stack.append(x)
    else:
        ops += ["Push", "Pop"]
    print(f"đọc {x}: stack = {stack}")
print(ops, len(ops), "=", 2 * target[-1] - len(target))

đọc 1: stack = []
đọc 2: stack = [2]
đọc 3: stack = [2, 3]
đọc 4: stack = [2, 3]
đọc 5: stack = [2, 3, 5]
['Push', 'Pop', 'Push', 'Push', 'Push', 'Pop', 'Push'] 7 = 7


## 5. Cách 1 — Brute Force / baseline (mô phỏng "thật")

**Ý tưởng:** làm đúng như mô tả: giữ một stack thật, đọc từng số của stream, `Push`; nếu số đó **không thuộc** `target` (kiểm tra bằng `x in target` trên list) thì `Pop`. Sau mỗi bước so **cả stack** với `target`; bằng rồi thì dừng.

Đây là cách "an toàn" nhất cho người mới vì bám sát luật, nhưng làm nhiều việc thừa: kiểm tra `in` trên list `O(m)` và so sánh list `O(m)` sau mỗi số.

In [8]:
def build_brute(target: List[int], n: int) -> List[str]:
    stack, ops = [], []
    for x in range(1, n + 1):
        stack.append(x)
        ops.append("Push")
        if x not in target:          # O(m) trên list
            stack.pop()
            ops.append("Pop")
        if stack == target:          # O(m) mỗi bước
            break
    return ops

for (target, n), want in examples:
    print(build_brute(target, n), build_brute(target, n) == want)

['Push', 'Push', 'Pop', 'Push'] True
['Push', 'Push', 'Push'] True
['Push', 'Push'] True


**Độ phức tạp:** Time `O(n·m)` với `m = len(target)` (mỗi số tốn `O(m)` cho `in` và `==`), Space `O(m)` cho stack (không tính output).

**Nút thắt:** không thực sự là vấn đề với `n ≤ 100`, nhưng có hai chỗ lãng phí: (1) không cần stack thật — chỉ cần biết đang chờ phần tử thứ mấy của `target`; (2) không cần so cả list — chỉ cần biết đã đọc tới `target[-1]` chưa.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — con trỏ thay cho stack:** vì `target` tăng dần, giữ chỉ số `i` = phần tử của `target` đang **chờ**. Đọc số `x`:
- **Nếu** `x == target[i]` **thì** `"Push"` và `i += 1`.
- **Ngược lại** (`x < target[i]`) **thì** `"Push","Pop"`.

**Insight 2 — dừng sớm:** stack bằng `target` **đúng lúc** vừa push `target[-1]`. Vậy chỉ cần đọc số `1..target[-1]`; `n` thực ra không ảnh hưởng gì (ràng buộc đảm bảo `target[-1] ≤ n`).

**Insight 3 — đếm theo khoảng trống:** giữa phần tử trước `prev` và phần tử hiện tại `t` có `t − prev − 1` số bị bỏ, mỗi số tốn `"Push","Pop"`, sau đó `t` tốn `"Push"`. Độ dài kết quả = `target[-1]` lần Push + `(target[-1] − m)` lần Pop = `2·target[-1] − m`.

**Invariant:** sau khi xử lý số `x`, stack (tưởng tượng) đúng bằng `target[0..i-1]` = các phần tử `target` ≤ `x`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Mô phỏng bằng con trỏ, dừng ở `target[-1]`

**Giải thích từng đoạn:**
1. `i = 0` — chờ `target[0]`.
2. Duyệt `x` từ `1` tới `target[-1]` (không tới `n`).
3. Luôn `"Push"`; nếu `x == target[i]` thì `i += 1`, ngược lại thêm `"Pop"`.

In [9]:
def build_pointer(target: List[int], n: int) -> List[str]:
    ops, i = [], 0
    for x in range(1, target[-1] + 1):
        ops.append("Push")
        if x == target[i]:
            i += 1
        else:
            ops.append("Pop")
    return ops

for (target, n), want in examples:
    print(build_pointer(target, n), build_pointer(target, n) == want)

['Push', 'Push', 'Pop', 'Push'] True
['Push', 'Push', 'Push'] True
['Push', 'Push'] True


**Code nộp LeetCode (cách 2):**

In [10]:
class SolutionPointer:
    def buildArray(self, target: List[int], n: int) -> List[str]:
        ops, i = [], 0
        for x in range(1, target[-1] + 1):
            ops.append("Push")
            if x == target[i]:
                i += 1
            else:
                ops.append("Pop")
        return ops

print(SolutionPointer().buildArray([2, 3, 5], 6))

['Push', 'Pop', 'Push', 'Push', 'Push', 'Pop', 'Push']


**Độ phức tạp:** Time `O(target[-1]) ⊆ O(n)`, Space `O(1)` ngoài output.

### 7.2 Cách 3 — Theo khoảng trống (gap)

**Giải thích từng đoạn:**
1. `prev = 0` (số cuối cùng đã đọc).
2. Với mỗi `t` trong `target`: thêm `["Push","Pop"] * (t − prev − 1)` cho các số bị bỏ, rồi thêm `"Push"` cho `t`.
3. Cập nhật `prev = t`. Vòng lặp tự dừng sau phần tử cuối — không cần nghĩ tới `n`.

In [11]:
def build_gap(target: List[int], n: int) -> List[str]:
    ops, prev = [], 0
    for t in target:
        ops += ["Push", "Pop"] * (t - prev - 1)
        ops.append("Push")
        prev = t
    return ops

for (target, n), want in examples:
    print(build_gap(target, n), build_gap(target, n) == want)

['Push', 'Push', 'Pop', 'Push'] True
['Push', 'Push', 'Push'] True
['Push', 'Push'] True


**Code nộp LeetCode (cách 3):**

In [12]:
class SolutionGap:
    def buildArray(self, target: List[int], n: int) -> List[str]:
        ops, prev = [], 0
        for t in target:
            ops += ["Push", "Pop"] * (t - prev - 1)
            ops.append("Push")
            prev = t
        return ops

print(SolutionGap().buildArray([1, 3], 3))

['Push', 'Push', 'Pop', 'Push']


**Độ phức tạp:** Time `O(m + |output|) = O(target[-1])`, Space `O(1)` ngoài output. Ưu điểm: số vòng lặp Python chỉ là `m`, phần lặp lại do phép nhân list làm ở tầng C.

### 7.3 Cách 4 — Mô phỏng stream bằng `iter`/`next` + `set` (phong cách "sự kiện")

Mô phỏng sát luật đề hơn: stream là iterator; ta đọc cho tới khi đã gom đủ `m` phần tử. Dùng `set` để kiểm tra thuộc `target` trong `O(1)`. Cách này hữu ích khi stream **không phải** `1..n` liên tục (xem biến thể ở mục 15).

In [13]:
def build_stream(target: List[int], n: int) -> List[str]:
    want = set(target)
    stream = iter(range(1, n + 1))
    ops, kept = [], 0
    while kept < len(target):
        x = next(stream)
        ops.append("Push")
        if x in want:
            kept += 1
        else:
            ops.append("Pop")
    return ops

class SolutionStream:
    def buildArray(self, target: List[int], n: int) -> List[str]:
        want, stream = set(target), iter(range(1, n + 1))
        ops, kept = [], 0
        while kept < len(target):
            ops.append("Push")
            if next(stream) in want:
                kept += 1
            else:
                ops.append("Pop")
        return ops

for (target, n), want in examples:
    print(build_stream(target, n) == want, SolutionStream().buildArray(target, n) == want)

True True
True True
True True


**Độ phức tạp:** Time `O(target[-1] + m)`, Space `O(m)` cho `set`.

## 8. TRACE từng bước

Trace cách 2 trên `target = [1,3,4,7]`, `n = 9`:

| x | target[i] chờ | x == target[i]? | thao tác | i sau | stack (tưởng tượng) |
|---|---|---|---|---|---|
| 1 | 1 | ✔ | Push | 1 | [1] |
| 2 | 3 | ✘ | Push, Pop | 1 | [1] |
| 3 | 3 | ✔ | Push | 2 | [1,3] |
| 4 | 4 | ✔ | Push | 3 | [1,3,4] |
| 5 | 7 | ✘ | Push, Pop | 3 | [1,3,4] |
| 6 | 7 | ✘ | Push, Pop | 3 | [1,3,4] |
| 7 | 7 | ✔ | Push | 4 | [1,3,4,7] → dừng |

In [14]:
def trace_build(target, n):
    ops, i, stack = [], 0, []
    for x in range(1, target[-1] + 1):
        waiting = target[i]
        ops.append("Push"); stack.append(x)
        if x == target[i]:
            i += 1
            act = "Push"
        else:
            ops.append("Pop"); stack.pop()
            act = "Push, Pop"
        print(f"x={x} chờ={waiting} -> {act:9} | i={i} stack={stack}")
    print("Dừng sau khi đọc", target[-1], "(không đọc tới n =", n, ")")
    return ops

res = trace_build([1, 3, 4, 7], 9)
print(res)

x=1 chờ=1 -> Push      | i=1 stack=[1]
x=2 chờ=3 -> Push, Pop | i=1 stack=[1]
x=3 chờ=3 -> Push      | i=2 stack=[1, 3]
x=4 chờ=4 -> Push      | i=3 stack=[1, 3, 4]
x=5 chờ=7 -> Push, Pop | i=3 stack=[1, 3, 4]
x=6 chờ=7 -> Push, Pop | i=3 stack=[1, 3, 4]
x=7 chờ=7 -> Push      | i=4 stack=[1, 3, 4, 7]
Dừng sau khi đọc 7 (không đọc tới n = 9 )
['Push', 'Push', 'Pop', 'Push', 'Push', 'Push', 'Pop', 'Push', 'Pop', 'Push']


Trace cách 3 (gap) trên cùng input để thấy mỗi phần tử `target` đóng góp bao nhiêu thao tác:

In [15]:
def trace_gap(target):
    ops, prev = [], 0
    for t in target:
        gap = t - prev - 1
        part = ["Push", "Pop"] * gap + ["Push"]
        ops += part
        print(f"t={t} prev={prev} gap={gap} -> thêm {part}")
        prev = t
    print("tổng", len(ops), "thao tác; công thức 2*target[-1]-m =", 2 * target[-1] - len(target))
    return ops

trace_gap([1, 3, 4, 7])

t=1 prev=0 gap=0 -> thêm ['Push']
t=3 prev=1 gap=1 -> thêm ['Push', 'Pop', 'Push']
t=4 prev=3 gap=0 -> thêm ['Push']
t=7 prev=4 gap=2 -> thêm ['Push', 'Pop', 'Push', 'Pop', 'Push']
tổng 10 thao tác; công thức 2*target[-1]-m = 10


['Push', 'Push', 'Pop', 'Push', 'Push', 'Push', 'Pop', 'Push', 'Pop', 'Push']

## 9. Code nộp LeetCode

Lời giải chính (cách 3 — theo khoảng trống), đúng chữ ký `buildArray(self, target, n)`. Tham số `n` không cần dùng vì luôn có `target[-1] ≤ n`.

In [16]:
from typing import List

class Solution:
    def buildArray(self, target: List[int], n: int) -> List[str]:
        ops, prev = [], 0
        for t in target:
            ops += ["Push", "Pop"] * (t - prev - 1)
            ops.append("Push")
            prev = t
        return ops

for (target, n), want in examples:
    assert Solution().buildArray(target, n) == want
print("Solution: 3 ví dụ đề OK")

Solution: 3 ví dụ đề OK


## 10. Vì sao thuật toán đúng

**Tính hợp lệ:** xét các số `1..target[-1]` theo thứ tự. Bằng quy nạp: trước khi đọc `x`, stack = các phần tử `target` nhỏ hơn `x`. Nếu `x ∈ target`, Push ⇒ stack = các phần tử `target` ≤ `x`. Nếu `x ∉ target`, Push rồi Pop ⇒ stack không đổi, vẫn = các phần tử `target` ≤ `x`. Sau `x = target[-1]`, stack = toàn bộ `target`.

**Dừng đúng lúc:** trước khi đọc `target[-1]`, stack thiếu ít nhất phần tử cuối nên chưa bằng `target` ⇒ chưa được phép dừng. Ngay sau khi push `target[-1]` thì stack bằng `target` ⇒ dừng, đúng luật "không đọc thêm".

**Không thể làm khác (gần như duy nhất):** một số `x ∉ target` đã push thì phải pop **trước** khi push số tiếp theo; nếu để lại, sau này muốn bỏ nó phải pop cả những số `target` đã nằm trên — mà stream không quay lại để push chúng lại được. Vì vậy mọi đáp án hợp lệ đều có dạng trên; ta vẫn test bằng **hàm kiểm tra hợp lệ** (mục 12) cho đúng tinh thần "trả về bất kỳ".

## 11. Phân tích Time / Space

Gọi `m = len(target)`, `T = target[-1] ≤ n`.

| Cách | Time | Space phụ | Độ dài output |
|---|---|---|---|
| 1. Mô phỏng thật | `O(T·m)` (dừng ở T) | `O(m)` | `2T − m` |
| 2. Con trỏ | `O(T)` | `O(1)` | `2T − m` |
| 3. Gap | `O(T)` (chỉ `m` vòng Python) | `O(1)` | `2T − m` |
| 4. Stream + set | `O(T + m)` | `O(m)` | `2T − m` |

Output luôn dài `Θ(T)` nên không cách nào nhanh hơn `O(T)`.

## 12. Bộ test

Thay vì chỉ so với đáp án cố định, ta viết **bộ kiểm tra hợp lệ** `is_valid(target, n, ops)` mô phỏng đúng luật: Push lấy số kế tiếp của stream (không vượt `n`), Pop cần stack không rỗng, kết thúc stack phải bằng `target`, và **không có thao tác nào sau khi** stack đã bằng `target`.

In [17]:
def is_valid(target, n, ops):
    stack, nxt = [], 1
    for k, op in enumerate(ops):
        if stack == target:
            return False                 # đã bằng target mà vẫn thao tác
        if op == "Push":
            if nxt > n:
                return False             # stream đã hết
            stack.append(nxt); nxt += 1
        elif op == "Pop":
            if not stack:
                return False
            stack.pop()
        else:
            return False                 # thao tác lạ
    return stack == target

print(is_valid([1, 3], 3, ["Push", "Push", "Pop", "Push"]))
print(is_valid([1, 2], 4, ["Push", "Push", "Push", "Pop"]), "<- đọc thêm số 3: sai luật")

True
False <- đọc thêm số 3: sai luật


In [18]:
ALL = {
    "brute": build_brute,
    "pointer": build_pointer,
    "gap": build_gap,
    "stream": build_stream,
    "Solution": lambda t, n: Solution().buildArray(t, n),
}
edge = [
    ([1], 1), ([1], 100), ([100], 100), ([5], 7),
    (list(range(1, 101)), 100), ([2, 4, 6, 8], 10), ([1, 100], 100), ([99, 100], 100),
]
for (target, n), want in examples:
    for name, f in ALL.items():
        assert f(target, n) == want, name
for target, n in edge:
    outs = [f(target, n) for f in ALL.values()]
    for o in outs:
        assert is_valid(target, n, o) and len(o) == 2 * target[-1] - len(target)
print("Ví dụ đề + edge case: OK")
print("[100], n=100 ->", len(build_gap([100], 100)), "thao tác")

Ví dụ đề + edge case: OK
[100], n=100 -> 199 thao tác


In [19]:
rng = random.Random(1441)
for _ in range(2000):
    n = rng.randint(1, 100)
    m = rng.randint(1, n)
    target = sorted(rng.sample(range(1, n + 1), m))
    base = build_brute(target, n)
    for name, f in ALL.items():
        out = f(target, n)
        assert is_valid(target, n, out), (name, target, n)
        assert out == base, name
print("2000 test ngẫu nhiên (seed 1441): mọi cách hợp lệ và trùng nhau")

2000 test ngẫu nhiên (seed 1441): mọi cách hợp lệ và trùng nhau


In [20]:
# Kiểm tra is_valid bắt được các đáp án sai thường gặp
target, n = [1, 3], 5
bad_cases = {
    "đọc tới n": ["Push", "Push", "Pop", "Push", "Push", "Pop", "Push", "Pop"],
    "quên Push trước Pop": ["Push", "Pop", "Push"],
    "thiếu phần tử": ["Push"],
    "pop stack rỗng": ["Pop", "Push"],
}
for name, ops in bad_cases.items():
    print(f"{name:22} -> hợp lệ? {is_valid(target, n, ops)}")

đọc tới n              -> hợp lệ? False
quên Push trước Pop    -> hợp lệ? False
thiếu phần tử          -> hợp lệ? False
pop stack rỗng         -> hợp lệ? False


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chạy tới `n` thay vì dừng ở `target[-1]`

Đề ví dụ 3 nhấn mạnh: đã bằng `target` thì không đọc thêm. Code sai duyệt hết `1..n`.

In [21]:
def wrong_until_n(target, n):
    tset, ops = set(target), []
    for x in range(1, n + 1):
        ops.append("Push")
        if x not in tset:
            ops.append("Pop")
    return ops

out = wrong_until_n([1, 2], 4)
print("Sai :", out, "| hợp lệ?", is_valid([1, 2], 4, out))
print("Đúng:", build_gap([1, 2], 4))

Sai : ['Push', 'Push', 'Push', 'Pop', 'Push', 'Pop'] | hợp lệ? False
Đúng: ['Push', 'Push']


**Sửa:** `for x in range(1, target[-1] + 1)` hoặc duyệt theo `target` (cách 3).

### Lỗi 2 — `ops += "Push"` làm tách chuỗi thành ký tự

`+=` trên list nhận **iterable**; chuỗi là iterable các ký tự.

In [22]:
def wrong_plus_string(target, n):
    ops, prev = [], 0
    for t in target:
        ops += ["Push", "Pop"] * (t - prev - 1)
        ops += "Push"            # SAI
        prev = t
    return ops

print("Sai :", wrong_plus_string([1, 3], 3))
print("Đúng:", build_gap([1, 3], 3))

Sai : ['P', 'u', 's', 'h', 'Push', 'Pop', 'P', 'u', 's', 'h']
Đúng: ['Push', 'Push', 'Pop', 'Push']


**Sửa:** `ops.append("Push")` hoặc `ops += ["Push"]` (có ngoặc vuông).

### Lỗi 3 — Số bị bỏ chỉ ghi `"Pop"` (quên đã phải `"Push"` nó trước)

Muốn bỏ một số khỏi stream, ta **bắt buộc** phải đọc (Push) nó trước; chỉ `"Pop"` là pop nhầm phần tử `target` đã có hoặc pop stack rỗng.

In [23]:
def wrong_only_pop(target, n):
    ops, i = [], 0
    for x in range(1, target[-1] + 1):
        if x == target[i]:
            ops.append("Push"); i += 1
        else:
            ops.append("Pop")    # SAI
    return ops

for target in ([1, 3], [2, 3]):
    out = wrong_only_pop(target, 3)
    print(f"target={target}: Sai {out} hợp lệ? {is_valid(target, 3, out)} | Đúng {build_gap(target, 3)}")

target=[1, 3]: Sai ['Push', 'Pop', 'Push'] hợp lệ? False | Đúng ['Push', 'Push', 'Pop', 'Push']
target=[2, 3]: Sai ['Pop', 'Push', 'Push'] hợp lệ? False | Đúng ['Push', 'Pop', 'Push', 'Push']


**Sửa:** số bị bỏ ⇒ `["Push", "Pop"]`.

### Lỗi 4 — Nhân list lồng nhau: `[["Push","Pop"]] * k`

Viết thừa một cặp ngoặc tạo list **các list**, không phải list phẳng các chuỗi.

In [24]:
def wrong_nested(target, n):
    ops, prev = [], 0
    for t in target:
        ops += [["Push", "Pop"]] * (t - prev - 1)   # SAI: phần tử là list
        ops.append("Push")
        prev = t
    return ops

out = wrong_nested([1, 4], 4)
print("Sai :", out, "| hợp lệ?", is_valid([1, 4], 4, out))
print("Đúng:", build_gap([1, 4], 4))

Sai : ['Push', ['Push', 'Pop'], ['Push', 'Pop'], 'Push'] | hợp lệ? False
Đúng: ['Push', 'Push', 'Pop', 'Push', 'Pop', 'Push']


**Sửa:** `ops += ["Push", "Pop"] * k` (một tầng ngoặc).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đề mô tả **quy trình từng bước** với luật rõ ràng ⇒ **simulation**: trước hết viết bản bám sát luật (dù chậm), rồi tìm chỗ thừa để rút gọn.
- Dãy **tăng nghiêm ngặt** + stream đi tới ⇒ dùng **con trỏ** vào dãy thay cho cấu trúc dữ liệu thật.
- Câu "dừng ngay khi …" ⇒ xác định **điểm dừng sớm** (ở đây là `target[-1]`), đừng mặc định chạy hết input.
- Luôn kiểm tra nhanh **độ dài output** bằng công thức (`2·target[-1] − m`) — cách tự kiểm rất mạnh.
- Viết hàm **validator** cho bài "trả về bất kỳ đáp án nào" — vừa dùng để test vừa giúp hiểu luật.

## 15. Biến thể của bài

1. **Stream tuỳ ý** (không phải `1..n`): `target` là dãy con của `stream`; cần kiểm tra khả thi (bài tập 5).
2. **Ngược lại:** cho danh sách thao tác, dựng lại `target` (bài tập 4).
3. **Validate Stack Sequences** (LeetCode 946): cho thứ tự push và thứ tự pop, hỏi có hợp lệ không — mô phỏng stack + con trỏ.
4. **Đếm số thao tác** mà không cần sinh danh sách: `2·target[-1] − m`.
5. **Hàng đợi (queue) thay vì stack:** bỏ số không cần thì phải "pop đầu" — nếu đã có phần tử `target` ở đầu thì không bỏ được; cần suy nghĩ lại luật.

In [25]:
def validate_stack_sequences(pushed, popped):
    # LeetCode 946 — minh hoạ biến thể cùng họ "mô phỏng stack + con trỏ"
    stack, j = [], 0
    for x in pushed:
        stack.append(x)
        while stack and j < len(popped) and stack[-1] == popped[j]:
            stack.pop(); j += 1
    return j == len(popped)

print(validate_stack_sequences([1, 2, 3, 4, 5], [4, 5, 3, 2, 1]))
print(validate_stack_sequences([1, 2, 3, 4, 5], [4, 3, 5, 1, 2]))

True
False


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space phụ | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng thật | Stack thật, `in` list, so cả stack | `O(T·m)` | `O(m)` | Khi mới đọc đề, để hiểu luật |
| 2. Con trỏ | `i` chờ `target[i]`, chạy `1..T` | `O(T)` | `O(1)` | Lời giải chuẩn, dễ giải thích |
| 3. Gap | `["Push","Pop"]*(t−prev−1)+["Push"]` | `O(T)` | `O(1)` | Code ngắn nhất |
| 4. Stream + set | `iter/next`, dừng khi gom đủ | `O(T+m)` | `O(m)` | Khi stream không liên tục |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output):** `buildArray([2,3,4], 4)` trả về gì?

**Bài 2 (tính tay):** không sinh danh sách, hãy tính số thao tác cho `target = [3, 10, 11, 50]`, `n = 60`; và số lần `"Pop"`.

**Bài 3 (điền TODO):** hoàn thiện `build_todo` (cách gap) bên dưới.

**Bài 4 (tự viết lại):** viết `ops_to_target(ops)` dựng lại `target` từ danh sách thao tác (giả sử stream là `1, 2, 3, …`).

**Bài 5 (mini challenge):** stream là một list tuỳ ý `stream` (các số **phân biệt**), `target` là list cần xây. Viết `build_from_stream(target, stream)` trả về danh sách thao tác theo luật tương tự, hoặc `None` nếu không thể (ví dụ `target` không phải dãy con của `stream`).

In [26]:
# Bài 3 — điền TODO (chỉ định nghĩa, không gọi)
def build_todo(target, n):
    ops, prev = [], 0
    for t in target:
        # TODO: thêm các cặp "Push","Pop" cho những số nằm giữa prev và t
        # TODO: thêm "Push" cho t
        # TODO: cập nhật prev
        pass
    return ops

## 18. Gợi ý 3 mức cho bài tập

**Bài 1:** (1) Số 1 có trong target không? (2) Số bị bỏ ⇒ Push, Pop. (3) Kết quả có `2·4 − 3 = 5` phần tử.

**Bài 2:** (1) Số Push = số lần đọc = `target[-1]`. (2) Số Pop = số bị bỏ = `target[-1] − m`. (3) Tổng `2·50 − 4`.

**Bài 3:** (1) Số lượng số nằm giữa là `t − prev − 1`. (2) Dùng phép nhân list. (3) `prev = t` ở cuối vòng.

**Bài 4:** (1) Giữ biến `nxt` = số sẽ được Push tiếp theo. (2) Push ⇒ `stack.append(nxt)`, `nxt += 1`. (3) Pop ⇒ `stack.pop()`; trả về `stack`.

**Bài 5:** (1) Duyệt `stream` với con trỏ `i` vào `target`. (2) Phần tử bằng `target[i]` ⇒ Push, `i += 1`; ngược lại ⇒ Push, Pop; dừng khi `i == len(target)`. (3) Hết stream mà `i < len(target)` ⇒ `None`.

## 19. Đáp án tham khảo cho bài tập

In [27]:
# Bài 1
ans1 = build_gap([2, 3, 4], 4)
print("Bài 1:", ans1)
assert ans1 == ["Push", "Pop", "Push", "Push", "Push"]

# Bài 2
t2 = [3, 10, 11, 50]
total, pops = 2 * t2[-1] - len(t2), t2[-1] - len(t2)
print("Bài 2: tổng", total, "| số Pop", pops)
assert total == len(build_gap(t2, 60)) == 96 and pops == build_gap(t2, 60).count("Pop") == 46

# Bài 3
def build_todo_sol(target, n):
    ops, prev = [], 0
    for t in target:
        ops += ["Push", "Pop"] * (t - prev - 1)
        ops.append("Push")
        prev = t
    return ops

for (target, n), want in examples:
    assert build_todo_sol(target, n) == want
print("Bài 3: OK")

Bài 1: ['Push', 'Pop', 'Push', 'Push', 'Push']
Bài 2: tổng 96 | số Pop 46
Bài 3: OK


In [28]:
# Bài 4
def ops_to_target(ops):
    stack, nxt = [], 1
    for op in ops:
        if op == "Push":
            stack.append(nxt); nxt += 1
        else:
            stack.pop()
    return stack

# Bài 5
def build_from_stream(target, stream):
    if not target:
        return []
    ops, i = [], 0
    for x in stream:
        ops.append("Push")
        if x == target[i]:
            i += 1
            if i == len(target):
                return ops
        else:
            ops.append("Pop")
    return None

def is_valid_stream(target, stream, ops):
    stack, k = [], 0
    for op in ops:
        if stack == target:
            return False
        if op == "Push":
            if k >= len(stream):
                return False
            stack.append(stream[k]); k += 1
        elif op == "Pop" and stack:
            stack.pop()
        else:
            return False
    return stack == target

assert ops_to_target(["Push", "Push", "Pop", "Push"]) == [1, 3]
assert build_from_stream([5, 2], [7, 5, 1, 2, 9]) == ["Push", "Pop", "Push", "Push", "Pop", "Push"]
assert build_from_stream([2, 5], [7, 5, 1, 2, 9]) is None

rng = random.Random(19)
for _ in range(800):
    n = rng.randint(1, 30)
    target = sorted(rng.sample(range(1, n + 1), rng.randint(1, n)))
    assert ops_to_target(build_gap(target, n)) == target
    stream = rng.sample(range(1, 40), rng.randint(1, 12))
    if rng.random() < 0.5:
        idx = sorted(rng.sample(range(len(stream)), rng.randint(1, len(stream))))
        tg = [stream[j] for j in idx]            # chắc chắn là dãy con
    else:
        tg = rng.sample(range(1, 40), rng.randint(1, 4))
    out = build_from_stream(tg, stream)
    it = iter(stream)
    is_subseq = all(any(v == w for w in it) for v in tg)
    assert (out is not None) == is_subseq
    if out is not None:
        assert is_valid_stream(tg, stream, out)
print("Bài 4, 5: OK (800 test ngẫu nhiên)")

Bài 4, 5: OK (800 test ngẫu nhiên)


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template gap:**

```python
ops, prev = [], 0
for t in target:
    ops += ["Push", "Pop"] * (t - prev - 1)
    ops.append("Push")
    prev = t
return ops
```

**Template con trỏ:**

```python
ops, i = [], 0
for x in range(1, target[-1] + 1):
    ops.append("Push")
    if x == target[i]: i += 1
    else: ops.append("Pop")
```

**Checklist:**
- [ ] Dừng ở `target[-1]`, không chạy tới `n`.
- [ ] Số bị bỏ ⇒ `"Push","Pop"` (đủ cả hai, đúng thứ tự).
- [ ] Không dùng `ops += "Push"` (thiếu ngoặc vuông).
- [ ] Độ dài kết quả bằng `2·target[-1] − len(target)`.
- [ ] Đã thử `target = [1]`, `target = [n]`, `target = [1..n]`.